In [45]:
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view

In [46]:
N = 1000
C,H_in,W_in = 1,28,28
m = 10
rng = np.random.default_rng(0)

In [47]:
from tensorflow.keras.datasets import mnist

In [48]:
def one_hot(y,num_classes=10):
    Y = np.zeros((num_classes,y.shape[0]),
                 dtype=np.float32)
    Y[y,np.arange(y.shape[0])] = 1.0
    return Y

In [49]:
(Xtr,Ytr), (Xte,Yte) = mnist.load_data()

Xtr = (Xtr.astype(np.float32)/255.0)[:,None,:,:] #(N,1,28,28)
Xte = (Xte.astype(np.float32)/255.0)[:,None,:,:]

idx = rng.permutation(Xtr.shape[0])
n_val = int(1e4)
val_idx = idx[:n_val]
train_idx = idx[n_val:]


X_train, y_train = Xtr[train_idx], Ytr[train_idx]
X_val,y_val = Xtr[val_idx], Ytr[val_idx]

Y_train = one_hot(y_train,10)
Y_val = one_hot(y_val,10)

In [50]:
#FORWARD

def relu_forward(Z):
    return np.maximum(0,Z)

def conv2_forward(X,W,b,pad=0,stride=1):
    if pad > 0: #(N,C,H+2p,W+2p)
        X_pad = np.pad(X, ((0,0),(0,0),(pad,pad),(pad,pad)), 
                       mode="constant")
    else:
        X_pad = X
                                           #(Kh,Kw)
    windows = sliding_window_view(X_pad,(W.shape[2],W.shape[3]),axis=(2,3))
    

    if stride > 1:
        windows = windows[:,:,::stride,::stride,:,:]
    
    Z = np.einsum("nchwkl,fckl->nfhw",windows,W,optimize=True) + b.reshape(1,-1,1,1)
    
    return Z, X_pad

def pool2d_forward(X,size=2,mode="max"):
    N,C,H,W = X.shape
    H_out, W_out = H//size, W//size
    H_crop, W_crop = H_out*size, W_out*size

    Xc = X[:,:,:H_crop,:W_crop]

    X_reshaped = Xc.reshape(N,C,H_out,size,W_out,size)

    if mode == "max":
        X_flat = X_reshaped.transpose(0,1,2,4,3,5).reshape(N,C,H_out,W_out,size*size)
        idx = np.argmax(X_flat,axis=-1) #(N,C,H_out,W_out)
        out = np.take_along_axis(X_flat,idx[...,None],axis=-1)[...,0]
        cache = ("max",X.shape,size,X_flat,idx)
        return out, cache

    else: #avg
        out = X_reshaped.mean(axis=(3,5))
        cache = ("avg",X.shape,size,X_reshaped)
        return out, cache

    

def flatten_forward(X):
    X_flat = X.reshape(X.shape[0],-1).T
    return X_flat, X.shape

def dense_forward(X,W,b):
    return W@X+b

In [51]:
#BACKWARD
def relu_backward(dA,Z):
    return dA*(Z>0)

def conv2d_backward(dZ,#(N,F,Ho,Wo)
                    X_pad,#(N,C,Hp,Wp)
                    W,#(F,C,Kh,Kw)
                    pad=0,stride=1):
    N,F,H_out,W_out = dZ.shape
    Fw,C,Kh,Kw = W.shape
    assert Fw==F

    db = np.sum(dZ,axis=(0,2,3)).reshape(F,1)

    #stride = 1
    H_full = X_pad.shape[2] - Kh + 1
    W_full = X_pad.shape[3] - Kw + 1

    if stride>1:
        dZ_dill = np.zeros((N,F,H_full,W_full))
        dZ_dill[:,:,::stride,::stride] = dZ
    else:
        dZ_dill = dZ

    X_windows = sliding_window_view(X_pad,(Kh,Kw),
                                           axis=(2,3))
    #X_windows: (N, C, H_full, W_full, Kh, Kw)
    dW = np.einsum("nfhw,nchwkl->fckl", dZ_dill, X_windows)

    dZ_pad = np.pad(dZ_dill,((0,0),(0,0),(Kh-1,Kh-1),(Kw-1,Kw-1)),
                    mode="constant")
    dZ_windows = sliding_window_view(dZ_pad,(Kh,Kw),axis=(2,3))
    #dZ_windows: (N, F, H_p, W_p, Kh, Kw)
    W_rot = W[:,:,::-1,::-1] #180 filters rot

    dX_pad = np.einsum("nfhwkl,fckl->nchw", dZ_windows, W_rot)
    dX = dX_pad[:, :, pad:-pad, pad:-pad] if pad > 0 else dX_pad
    #(N,C,H,W)
    return dX, dW, db

def pool2d_backward(dout,cache):
    mode = cache[0]
    N, C, _, _ = dout.shape
    
    if mode == "avg":
        _,X_shape,size,X_resh = cache
        H_out,W_out = dout.shape[2],dout.shape[3]
        H_crop,W_crop = H_out*size,W_out*size
    
        dout_resh = dout.reshape(N,C,H_out,1,W_out,1)
        dX_resh=np.ones_like(X_resh)*(dout_resh/(size*size))
        dX_crop = dX_resh.reshape(N,C,H_crop,W_crop)
    else: #max
        _, X_shape, size, X_flat, idx = cache
        H_out,W_out = dout.shape[2],dout.shape[3]
        H_crop,W_crop = H_out*size,W_out*size

        assert X_flat.shape == (N, C, H_out, W_out, size*size)
        assert idx.shape == (N, C, H_out, W_out)

        dX_flat = np.zeros_like(X_flat)
        np.put_along_axis(dX_flat,idx[...,None],dout[...,None],axis=-1)
        dX_resh = dX_flat.reshape(N,C,H_out,W_out,size,size).transpose(0,1,2,4,3,5)
        dX_crop = dX_resh.reshape(N,C,H_crop,W_crop)

    dX = np.zeros(X_shape,dtype=dout.dtype)
    dX[:,:,:dX_crop.shape[2],:dX_crop.shape[3]] = dX_crop
    return dX


def flatten_backward(dZ, orig_shape):
    return dZ.T.reshape(orig_shape)

def dense_backward(dZ,X,W):
    dX = W.T@dZ
    dW = dZ@X.T
    db=np.sum(dZ,axis=1,keepdims=True)
    return dX,dW,db

In [52]:
def init_network(arch,input_shape,rng):
    params = [None]*len(arch)
    current_shape = input_shape

    for l, layer in enumerate(arch):
        l_type = layer["type"]

        if l_type=="conv":
            C_in,H,W = current_shape
            F,K = layer["filters"],layer["kernel"]
            pad,stride=layer.get("pad",0),layer.get("stride",1)

            std=np.sqrt(2.0/(C_in*K*K))
            params[l]={
                "W":rng.normal(0.0,std,size=(F,C_in,K,K)),
                "b":np.zeros((F,1))
            }
            current_shape = (F,(H+2*pad-K)//stride+1,
                             (W+2*pad-K)//stride+1)
            
        elif l_type == "pool":
            C, H, W = current_shape
            size = layer.get("size", 2)
            current_shape = (C, H // size, W // size)
            
        elif l_type == "flatten":
            current_shape = (np.prod(current_shape),)
            
        elif l_type == "dense":
            d_in, d_out = current_shape[0], layer["units"]
            std = np.sqrt(2.0 / d_in)
            params[l] = {
                "W": rng.normal(0.0, std, size=(d_out, d_in)),
                "b": np.zeros((d_out, 1))
            }
            current_shape = (d_out,)
            
    return params

In [53]:
def forward_net(X, arch, params):
    cache =[]
    A = X
    
    for l, layer in enumerate(arch):
        l_type = layer["type"]
        if l_type == "conv":
            Z, X_pad = conv2_forward(A, params[l]["W"], params[l]["b"], 
                                      layer.get("pad",0), layer.get("stride",1))
            cache.append((X_pad, Z))
            A = Z
        elif l_type == "relu":
            cache.append(A)
            A = relu_forward(A)
        elif l_type == "pool":
            P, pool_cache = pool2d_forward(A, 
                            size=layer.get("size",2),
                            mode=layer.get("mode","max"))
            cache.append(pool_cache)
            A = P
        elif l_type == "flatten":
            F, orig_shape = flatten_forward(A)
            cache.append(orig_shape)
            A = F
        elif l_type == "dense":
            Z = dense_forward(A, params[l]["W"], params[l]["b"])
            cache.append((A, Z))
            A = Z
            
    return A, cache

In [54]:
def backward_net(dA, cache, arch, params):
    grads = [None] * len(arch)
    
    for l in reversed(range(len(arch))):
        l_type, layer = arch[l]["type"], arch[l]
        
        if l_type == "dense":
            X_prev, Z = cache[l]
            dA, dW, db = dense_backward(dA, X_prev, params[l]["W"])
            grads[l] = {"W": dW, "b": db}
        elif l_type == "flatten":
            dA = flatten_backward(dA, cache[l])
        elif l_type == "pool":
            dA = pool2d_backward(dA, cache[l])
        elif l_type == "relu":
            dA = relu_backward(dA, cache[l])
        elif l_type == "conv":
            X_pad, Z = cache[l]
            dA, dW, db = conv2d_backward(dA, X_pad, params[l]["W"], 
                                         layer.get("pad",0), layer.get("stride",1))
            grads[l] = {"W": dW, "b": db}
            
    return grads

def update_net(params, grads, lr):
        for p, g in zip(params, grads):
            if p is not None and g is not None:
                if p is not None:
                    p["W"] -= lr * g["W"]
                    p["b"] -= lr * g["b"]
        return params

In [55]:
def softmax(Z):
    Zs = Z - np.max(Z,axis=0,keepdims=True)
    expZ = np.exp(Zs)
    return expZ/np.sum(expZ,axis=0,keepdims=True)

def loss_ce_from_logits(logits,Y):
    P = softmax(logits)
    eps = 1e-12
    loss = -np.mean(np.sum(Y*np.log(P+eps),axis=0))
    return loss,P

In [56]:
def fit(X_train, Y_train, arch, params, lr=2e-2, epochs=5, batch_size=64, X_val=None, 
        Y_val=None, y_val_int=None):
    N_samples = X_train.shape[0]
    history = {"train_loss": [], "val_loss": [], "val_acc": []}

    for ep in range(1, epochs + 1):
        idx = np.random.permutation(N_samples)
        train_losses =[]

        for start in range(0, N_samples, batch_size):
            batch_idx = idx[start:start+batch_size]
            Xb, Yb = X_train[batch_idx], Y_train[:, batch_idx]

            #Forward -> Loss -> Backward -> Update
            logits, cache = forward_net(Xb, arch, params)
            L,P = loss_ce_from_logits(logits, Yb)
            dA0 = (P-Yb)/Xb.shape[0]
            grads = backward_net(dA0, cache, arch, params)
            params = update_net(params, grads, lr)
            
            train_losses.append(L)

        train_loss = float(np.mean(train_losses))
        history["train_loss"].append(train_loss)

        msg = f"Epoch {ep}/{epochs} | Train CE: {train_loss:.4f}"

        if X_val is not None and Y_val is not None:
            logits_val, _ = forward_net(X_val, arch, params)
            val_loss,val_P = loss_ce_from_logits(logits_val,Y_val)
            history["val_loss"].append(float(val_loss))

            if y_val_int is not None:
                pred = np.argmax(val_P,axis=0)
                acc = float(np.mean(pred == y_val_int))
                history["val_acc"].append(acc)
                msg += f" | Val CE: {val_loss:.4f} | Val Acc: {acc:.4f}"
            else:
                msg += f" | Val CE: {val_loss:.4f}"

        print(msg)
            
    return params, history

In [57]:
arch =[
    # Input: 1 x 28 x 28
    {"type": "conv", "filters": 4, "kernel": 3, "pad": 1, "stride": 1}, # -> 4x28x28
    {"type": "relu"},
    {"type": "pool", "mode": "max", "size": 2},                         # -> 4x14x14
    
    {"type": "conv", "filters": 8, "kernel": 3, "pad": 1, "stride": 1}, # -> 8x14x14
    {"type": "relu"},
    {"type": "pool", "mode": "max", "size": 2},                         # -> 8x7x7
    
    {"type": "flatten"},                                                # 8*7*7 = 392
    {"type": "dense", "units": 64},
    {"type": "relu"},
    {"type": "dense", "units": m}                                       
]

In [58]:
params = init_network(arch, input_shape=(C,H_in,W_in), rng=rng)
params, history = fit(X_train, Y_train, arch, params,
                      lr=2e-2, epochs=5, batch_size=64,
                      X_val=X_val, Y_val=Y_val, y_val_int=y_val)

Epoch 1/5 | Train CE: 0.6197 | Val CE: 0.3493 | Val Acc: 0.8875
Epoch 2/5 | Train CE: 0.1859 | Val CE: 0.1816 | Val Acc: 0.9445
Epoch 3/5 | Train CE: 0.1434 | Val CE: 0.1401 | Val Acc: 0.9584
Epoch 4/5 | Train CE: 0.1215 | Val CE: 0.1732 | Val Acc: 0.9480
Epoch 5/5 | Train CE: 0.1054 | Val CE: 0.1146 | Val Acc: 0.9658
